# dotLLM — lm_head / output-routed experiment (T4 x2)

Trains `lm_head` + converted final transformer block(s) (layers 26-29) on the EN+JA mix
to convergence. Uses Knowledge Distillation with the frozen teacher on `cuda:1` (T4 x2
split) by default; KD can be disabled via the CONFIG cell.

## Quick-start checklist

| Step | Where |
|------|-------|
| 1. Accelerator → **GPU T4 x2** | Settings → Accelerator |
| 2. Internet → **On** | Settings → Internet |
| 3. Add secret **`GITHUB_PAT`** (repo write) | Settings → Secrets |
| 4. Edit CONFIG cell if needed | Cell 2 |
| 5. **Run All** (interactive) OR **Save & Run All (Commit)** for multi-session | — |

## Multi-session auto-resume (runs > ~9 h)

Kaggle's interactive sessions have a ~9-12 h wall-clock limit.  This notebook
checkpoints every `CHECKPOINT_EVERY` steps to `/kaggle/working/ckpt/ja_lmhead/checkpoint/`.

For runs that exceed one session:

1. Run as **Save & Run All (Commit)** — Kaggle saves `/kaggle/working/` as the
   version's output dataset.
2. After the version finishes (or times out), open a **new version** of this notebook.
3. Under **Data** → **Add Data** → search for this notebook's output dataset → attach it
   as input.  The checkpoint will appear under `/kaggle/input/<dataset-slug>/ckpt/ja_lmhead/checkpoint/`.
4. Cell 7 auto-detects the checkpoint and passes `--resume-from` to `mote_train.py`.

> **Note:** Interactive sessions do **not** persist `/kaggle/working/` across restarts.
> Only **Commit** runs produce a persistent output dataset.

## Write-check

Cell 5 performs a real git push to `kaggle-results` branch **before** any training
starts.  If the push fails the cell raises an error and execution stops — you will
not waste GPU hours only to discover the push is broken at the end.

In [ ]:
# ── CONFIG — edit before running ────────────────────────────────────────────
# All tunable parameters are here.  All other cells are fixed.

RESULTS_REMOTE   = "https://github.com/jamesburton/dotLLM"
REPO_URL         = "https://github.com/jamesburton/dotLLM.git"
BRANCH           = "issue/trackM-mote"
REPO_DIR         = "/kaggle/working/dotLLM"

# ── Experiment ──────────────────────────────────────────────────────────────
MIX              = "ja_en"   # data mix: EN + JA held-out eval
N_EXPERTS        = 4
TOP_K            = 1
SHARED           = "none"
LAYERS           = "26-29"   # final transformer blocks to convert to MoTE
TOKENS           = "2e7"     # token budget (20 M).  Raise for longer runs.
SEQ_LEN          = 512
EVAL_EVERY       = 500       # print JA/EN held-out curve every N steps
CHECKPOINT_EVERY = 1000      # checkpoint to /kaggle/working/ every N steps

# ── Knowledge Distillation ──────────────────────────────────────────────────
# KD_ON=True  → teacher on cuda:1, student on cuda:0 (T4x2 split, recommended)
# KD_ON=False → LM-only loss, no teacher needed (faster; ~0.5 ppl worse)
KD_ON            = True
KD_WEIGHT        = 0.5       # weight on teacher KL loss; ignored when KD_ON=False
TEACHER_DEVICE   = "cuda:1"  # T4x2: teacher on second card.  Change to "cpu" if needed.

# ── Paths ───────────────────────────────────────────────────────────────────
# /kaggle/working/ persists as the version output when running as a Commit.
OUT_DIR          = "/kaggle/working/ckpt/ja_lmhead"   # adapter weights + metrics here
CKPT_DIR         = OUT_DIR + "/checkpoint"             # --resume-from target
EXPERIMENT_ID    = "lmhead_ja"                         # results/<id>/ on kaggle-results

print(f"CONFIG loaded: MIX={MIX!r}  LAYERS={LAYERS!r}  TOKENS={TOKENS}")
print(f"KD={'ON (weight=' + str(KD_WEIGHT) + ', teacher=' + TEACHER_DEVICE + ')' if KD_ON else 'OFF (LM-only)'}")
print(f"OUT_DIR={OUT_DIR!r}  CKPT_DIR={CKPT_DIR!r}")

In [ ]:
# ── Cell 2: install dependencies ────────────────────────────────────────────
# Re-run after any kernel restart.
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "transformers", "peft", "datasets", "accelerate", "bitsandbytes"],
    check=True,
)
print("Dependencies ready.")

In [ ]:
# ── Cell 3: load GITHUB_PAT from Kaggle Secrets ─────────────────────────────
# Add the secret: notebook Settings → Secrets → Add New Secret
#   Name:  GITHUB_PAT
#   Value: your GitHub PAT (ghp_...) with 'repo' Contents: write scope
import os

try:
    from kaggle_secrets import UserSecretsClient
    _pat = UserSecretsClient().get_secret("GITHUB_PAT")
    os.environ["GITHUB_PAT"] = _pat
    os.environ["GH_PAT"] = _pat  # fallback for push_results.py
    print(f"GITHUB_PAT loaded from Kaggle Secrets ({len(_pat)} chars)")
except Exception as _e:
    # Try GH_PAT as secret name fallback
    try:
        from kaggle_secrets import UserSecretsClient
        _pat = UserSecretsClient().get_secret("GH_PAT")
        os.environ["GITHUB_PAT"] = _pat
        os.environ["GH_PAT"] = _pat
        print(f"GH_PAT loaded from Kaggle Secrets as fallback ({len(_pat)} chars)")
    except Exception as _e2:
        print(f"WARNING: could not load GITHUB_PAT from Kaggle Secrets: {_e}")
        print("Set manually: os.environ['GITHUB_PAT'] = 'ghp_...'")

print(f"GITHUB_PAT set: {bool(os.environ.get('GITHUB_PAT'))}")

In [ ]:
# ── Cell 4: git write-check ─────────────────────────────────────────────────
# Push a tiny sentinel file to the kaggle-results branch RIGHT NOW,
# before any training.  If this fails, we abort immediately so we never
# waste GPU hours and then discover the push is broken at the end.
import datetime, os, shutil, subprocess, sys, tempfile


def _writecheck(pat: str, results_remote: str) -> bool:
    """Push results/_writecheck/<ts>.txt to kaggle-results. Return True on success."""
    ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    auth = (
        results_remote.replace("https://", f"https://{pat}@", 1)
        if results_remote.startswith("https://")
        else results_remote
    )
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    tmpdir = tempfile.mkdtemp(prefix="dotllm_wcheck_")
    repo = os.path.join(tmpdir, "repo")
    try:
        # Clone kaggle-results branch (create it if it does not exist yet)
        try:
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", "kaggle-results",
                 auth, repo],
                check=True, capture_output=True, env=env,
            )
        except subprocess.CalledProcessError:
            subprocess.run(
                ["git", "clone", "--depth", "1", auth, repo],
                check=True, capture_output=True, env=env,
            )
            subprocess.run(
                ["git", "-C", repo, "checkout", "-b", "kaggle-results"],
                check=True,
            )

        check_dir = os.path.join(repo, "results", "_writecheck")
        os.makedirs(check_dir, exist_ok=True)
        with open(os.path.join(check_dir, f"{ts}.txt"), "w", encoding="utf-8") as fh:
            fh.write(f"write-check at {ts}\n")

        subprocess.run(["git", "-C", repo, "config", "user.email",
                        "kaggle-bot@dotllm.dev"], check=True)
        subprocess.run(["git", "-C", repo, "config", "user.name",
                        "dotLLM Kaggle Bot"], check=True)
        subprocess.run(["git", "-C", repo, "add", "-A"], check=True)
        subprocess.run(["git", "-C", repo, "commit", "-m", f"writecheck: {ts}"],
                       check=True)
        subprocess.run(
            ["git", "-C", repo, "push", auth, "HEAD:kaggle-results"],
            check=True, env=env,
        )
        print(f"[writecheck] PASSED — pushed results/_writecheck/{ts}.txt")
        return True
    except subprocess.CalledProcessError as exc:
        print(f"[writecheck] FAILED: {exc}", file=sys.stderr)
        return False
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


_pat = os.environ.get("GITHUB_PAT") or os.environ.get("GH_PAT", "")
if not _pat:
    raise RuntimeError(
        "GITHUB_PAT not set. Run Cell 3 first, or set os.environ['GITHUB_PAT'] manually."
    )

print("[writecheck] Verifying git write access to kaggle-results branch...")
_ok = _writecheck(_pat, RESULTS_REMOTE)
if not _ok:
    raise RuntimeError(
        "\n"
        "╔══════════════════════════════════════════════════════════╗\n"
        "║  GIT WRITE-CHECK FAILED — ABORTING                      ║\n"
        "║  Cannot push to the kaggle-results branch.              ║\n"
        "║  Fix ONE of these before proceeding:                    ║\n"
        "║    1. GITHUB_PAT needs 'repo' Contents: write scope     ║\n"
        "║    2. Token may be expired — regenerate at github.com   ║\n"
        "║    3. Enable Internet: Settings → Internet → On         ║\n"
        "║  DO NOT proceed — training would fail on the push step. ║\n"
        "╚══════════════════════════════════════════════════════════╝"
    )
print("[writecheck] Write access confirmed — safe to proceed.")

In [ ]:
# ── Cell 5: clone the repo ──────────────────────────────────────────────────
import os, subprocess

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Cloning {REPO_URL} @ {BRANCH} ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        check=True,
    )
else:
    print("Repo already present — pulling latest ...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print(f"Repo ready at {REPO_DIR}  (cwd={os.getcwd()})")

In [ ]:
# ── Cell 6: detect checkpoint / auto-resume ──────────────────────────────────
# Search order:
#   1. CKPT_DIR (same-session working dir — survives within one Commit run)
#   2. /kaggle/input/*/ (prior version's output attached as a dataset input)
#
# For runs that span >1 session:
#   • Run as "Save & Run All (Commit)" so /kaggle/working/ is saved as version output.
#   • In the next version, attach the previous version's output as Data input.
#   • This cell will find state.json under /kaggle/input/<dataset-slug>/ckpt/ja_lmhead/checkpoint/
#     and set RESUME_FROM automatically.
import glob, json, os

RESUME_FROM = None

# 1. Same-session
if os.path.isfile(os.path.join(CKPT_DIR, "state.json")):
    RESUME_FROM = CKPT_DIR
    print(f"[resume] Found same-session checkpoint: {CKPT_DIR}")
else:
    # 2. Cross-session: prior Commit version attached as dataset input
    _patterns = [
        "/kaggle/input/*/ckpt/ja_lmhead/checkpoint/state.json",
        "/kaggle/input/*/checkpoint/state.json",
        "/kaggle/input/*/state.json",
    ]
    _candidates = [hit for pat in _patterns for hit in glob.glob(pat)]
    if _candidates:
        _best = max(_candidates, key=os.path.getmtime)
        RESUME_FROM = os.path.dirname(_best)
        print(f"[resume] Found cross-session checkpoint (attached dataset): {RESUME_FROM}")
    else:
        print("[resume] No prior checkpoint found — training from scratch")

if RESUME_FROM:
    try:
        with open(os.path.join(RESUME_FROM, "state.json"), encoding="utf-8") as _f:
            _st = json.load(_f)
        print(
            f"[resume] Resuming from step={_st.get('step', 0):,}, "
            f"tokens_seen={_st.get('tokens_seen', 0):,}"
        )
    except Exception as _e:
        print(f"[resume] WARNING: could not read state.json: {_e}")
else:
    print("[resume] Starting fresh run")

In [ ]:
# ── Cell 7: run mote_train.py ────────────────────────────────────────────────
import os, subprocess, sys

os.makedirs(OUT_DIR, exist_ok=True)

train_cmd = [
    sys.executable,
    os.path.join(REPO_DIR, "scripts", "lora", "mote_train.py"),
    "--mix",              MIX,
    "--n-experts",        str(N_EXPERTS),
    "--top-k",            str(TOP_K),
    "--shared",           SHARED,
    "--layers",           LAYERS,
    "--train-lm-head",
    "--grad-checkpoint",
    "--eval-every",       str(EVAL_EVERY),
    "--tokens",           TOKENS,
    "--device",           "cuda",
    "--optim",            "adamw8bit",
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--out",              OUT_DIR,
]

if KD_ON:
    train_cmd += ["--kd-weight", str(KD_WEIGHT), "--teacher-device", TEACHER_DEVICE]

if RESUME_FROM:
    train_cmd += ["--resume-from", RESUME_FROM]

print("Training command:")
print("  " + " ".join(train_cmd))
print()

# Stream output live so eval curves appear in real time
_proc = subprocess.Popen(
    train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
assert _proc.stdout is not None
for _line in _proc.stdout:
    print(_line, end="", flush=True)
_proc.wait()

if _proc.returncode != 0:
    raise RuntimeError(
        f"mote_train.py exited with code {_proc.returncode}. "
        "Check output above for CUDA OOM or data errors."
    )
print("\n[train] Training complete.")

In [ ]:
# ── Cell 8: push results to kaggle-results branch ─────────────────────────────
# Pushes metrics.json + eval.json + eval_curve + mote_config.json to
# results/lmhead_ja/ on the kaggle-results branch.
# Uses --train-artifacts-only so push_results.py skips the grid_manifest.json
# status-flip (this experiment has no manifest entry).
import os, subprocess, sys

push_cmd = [
    sys.executable,
    os.path.join(REPO_DIR, "kaggle", "push_results.py"),
    "--cell-id",         EXPERIMENT_ID,
    "--adapter",         OUT_DIR,
    "--results-remote",  RESULTS_REMOTE,
    "--train-artifacts-only",  # no manifest entry for this experiment
]

print("Push command:")
print("  " + " ".join(push_cmd))
print()

subprocess.run(push_cmd, check=True)
print(f"\n[push] Results pushed to kaggle-results/results/{EXPERIMENT_ID}/")
print("All done.")